In [ ]:
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
import json

with open('circles.json') as f:
    data = json.load(f)

iterations = list(range(1, len(data) + 1))
lats = [d['lat'] for d in data]
lons = [d['lon'] for d in data]
radii = [d['radius_m'] for d in data]

delta_lat = [0] + [lats[i] - lats[i-1] for i in range(1, len(lats))]
delta_lon = [0] + [lons[i] - lons[i-1] for i in range(1, len(lons))]
delta_radius = [0] + [radii[i] - radii[i-1] for i in range(1, len(radii))]

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
fig.suptitle('Circle Movement & Shrinkage Over Time', fontsize=14)

axes[0, 0].plot(iterations, lats, color='steelblue')
axes[0, 0].set_title('Latitude')
axes[0, 0].set_ylabel('degrees')

axes[1, 0].plot(iterations, lons, color='darkorange')
axes[1, 0].set_title('Longitude')
axes[1, 0].set_ylabel('degrees')

axes[2, 0].plot(iterations, radii, color='seagreen')
axes[2, 0].set_title('Radius')
axes[2, 0].set_ylabel('meters')
axes[2, 0].set_xlabel('Iteration')

axes[0, 1].plot(iterations, delta_lat, color='steelblue')
axes[0, 1].axhline(0, color='gray', linewidth=0.5, linestyle='--')
axes[0, 1].set_title('Δ Latitude')
axes[0, 1].set_ylabel('degrees / step')

axes[1, 1].plot(iterations, delta_lon, color='darkorange')
axes[1, 1].axhline(0, color='gray', linewidth=0.5, linestyle='--')
axes[1, 1].set_title('Δ Longitude')
axes[1, 1].set_ylabel('degrees / step')

axes[2, 1].plot(iterations, delta_radius, color='seagreen')
axes[2, 1].axhline(0, color='gray', linewidth=0.5, linestyle='--')
axes[2, 1].set_title('Δ Radius')
axes[2, 1].set_ylabel('meters / step')
axes[2, 1].set_xlabel('Iteration')

for ax in axes.flat:
    ax.set_xlabel('Iteration')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from IPython.display import HTML

df = pd.DataFrame({
    'Iteration': iterations,
    'Latitude': lats,
    'Longitude': lons,
    'Radius (m)': radii,
    'Δ Latitude': delta_lat,
    'Δ Longitude': delta_lon,
    'Δ Radius (m)': delta_radius,
})
df.set_index('Iteration', inplace=True)

html = f'<div style="height:500px;overflow-y:auto;overflow-x:auto;">{df.to_html()}</div>'
display(HTML(html))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

start = 455  # iteration 456 (0-indexed)
iters_seg = np.array(iterations[start:])
lats_seg  = np.array(lats[start:])
lons_seg  = np.array(lons[start:])
radii_seg = np.array(radii[start:])

# Radius: linear (constant step) → find where it hits 0
rad_coeffs   = np.polyfit(iters_seg, radii_seg, 1)
converge_iter = int(-rad_coeffs[1] / rad_coeffs[0])

# Lat & lon: deltas decrease linearly → quadratic fit to actual values
lat_coeffs = np.polyfit(iters_seg, lats_seg, 2)
lon_coeffs = np.polyfit(iters_seg, lons_seg, 2)

iters_proj = np.arange(iters_seg[0], converge_iter + 1)
lat_proj   = np.polyval(lat_coeffs, iters_proj)
lon_proj   = np.polyval(lon_coeffs, iters_proj)
rad_proj   = np.polyval(rad_coeffs, iters_proj)

converge_lat = np.polyval(lat_coeffs, converge_iter)
converge_lon = np.polyval(lon_coeffs, converge_iter)

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle(
    f'Convergence Projection (quadratic lat/lon, linear radius)\n'
    f'Radius → 0 at iteration {converge_iter}  |  '
    f'lat {converge_lat:.5f}°   lon {converge_lon:.5f}°',
    fontsize=11
)

for ax, actual, proj, label, color in zip(
    axes,
    [lats_seg, lons_seg, radii_seg],
    [lat_proj,  lon_proj,  rad_proj],
    ['Latitude (°)', 'Longitude (°)', 'Radius (m)'],
    ['steelblue', 'darkorange', 'seagreen'],
):
    ax.plot(iters_seg, actual, color=color, linewidth=2, label='Actual')
    ax.plot(iters_proj, proj, color=color, linewidth=1.2,
            linestyle='--', alpha=0.6, label='Projected (quadratic)')
    ax.axvline(converge_iter, color='red', linewidth=0.8,
               linestyle=':', label=f'Convergence iter {converge_iter}')
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

axes[-1].set_xlabel('Iteration')
axes[-1].axhline(0, color='red', linewidth=0.8, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

# Also show the delta curves to confirm the linear-decay assumption
fig2, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
fig2.suptitle('Δ Latitude & Δ Longitude per step (456→503) — linear decay confirms quadratic model')

d_iters = iters_seg[1:]
d_lat = np.diff(lats_seg)
d_lon = np.diff(lons_seg)
dlat_fit = np.polyfit(d_iters, d_lat, 1)
dlon_fit = np.polyfit(d_iters, d_lon, 1)

ax1.plot(d_iters, d_lat, color='steelblue', label='Δ lat actual')
ax1.plot(d_iters, np.polyval(dlat_fit, d_iters), 'k--', linewidth=1, label='linear fit')
ax1.set_ylabel('Δ Latitude'); ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

ax2.plot(d_iters, d_lon, color='darkorange', label='Δ lon actual')
ax2.plot(d_iters, np.polyval(dlon_fit, d_iters), 'k--', linewidth=1, label='linear fit')
ax2.set_ylabel('Δ Longitude'); ax2.set_xlabel('Iteration')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Convergence iteration : {converge_iter}")
print(f"Converged latitude    : {converge_lat:.6f}°")
print(f"Converged longitude   : {converge_lon:.6f}°")
